# Sales & Demand Forecasting - Model Training
## Future Interns - Machine Learning Task 1

This notebook trains forecasting models on real business datasets and saves them for deployment.

## 1. Install Required Libraries

In [ ]:
!pip install kagglehub pandas numpy scikit-learn matplotlib seaborn joblib

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 3. Load Dataset

### Option 1: Kaggle Superstore Dataset

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load Superstore dataset
file_path = "Sample - Superstore.csv"
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "vivek468/superstore-dataset-final",
    file_path
)

print("Dataset Shape:", df.shape)
print("\nFirst 5 records:")
df.head()

### Option 2: UCI Online Retail Dataset

In [ ]:
# Uncomment to use UCI dataset instead
# !pip install ucimlrepo
# from ucimlrepo import fetch_ucirepo
# 
# online_retail = fetch_ucirepo(id=352)
# X = online_retail.data.features
# y = online_retail.data.targets
# df = pd.concat([X, y], axis=1)
# print(online_retail.metadata)
# print(online_retail.variables)

## 4. Data Exploration & Cleaning

In [ ]:
# Check data info
print("Dataset Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nBasic Statistics:")
df.describe()

In [ ]:
# Clean data
# Convert Order Date to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Remove any missing values
df = df.dropna()

# Sort by date
df = df.sort_values('Order Date')

print(f"Cleaned dataset shape: {df.shape}")
print(f"Date range: {df['Order Date'].min()} to {df['Order Date'].max()}")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Sales over time
daily_sales = df.groupby('Order Date')['Sales'].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(daily_sales['Order Date'], daily_sales['Sales'], alpha=0.7)
plt.title('Daily Sales Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Sales by Category
category_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
category_sales.plot(kind='bar', color=['#667eea', '#764ba2', '#f093fb'])
plt.title('Total Sales by Category', fontsize=16, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Sales ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly sales trend
df['Year-Month'] = df['Order Date'].dt.to_period('M')
monthly_sales = df.groupby('Year-Month')['Sales'].sum()

plt.figure(figsize=(14, 6))
monthly_sales.plot(kind='line', marker='o', color='#667eea', linewidth=2)
plt.title('Monthly Sales Trend', fontsize=16, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Sales ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
# Aggregate to daily level
daily_data = df.groupby('Order Date').agg({
    'Sales': 'sum',
    'Quantity': 'sum',
    'Profit': 'sum'
}).reset_index()

# Create time-based features
daily_data['DayOfWeek'] = daily_data['Order Date'].dt.dayofweek
daily_data['Month'] = daily_data['Order Date'].dt.month
daily_data['Quarter'] = daily_data['Order Date'].dt.quarter
daily_data['Year'] = daily_data['Order Date'].dt.year
daily_data['DayOfYear'] = daily_data['Order Date'].dt.dayofyear
daily_data['WeekOfYear'] = daily_data['Order Date'].dt.isocalendar().week

# Create lag features (previous days' sales)
for lag in [1, 7, 14, 30]:
    daily_data[f'Sales_Lag_{lag}'] = daily_data['Sales'].shift(lag)

# Rolling averages
daily_data['Sales_MA_7'] = daily_data['Sales'].rolling(window=7).mean()
daily_data['Sales_MA_30'] = daily_data['Sales'].rolling(window=30).mean()

# Drop rows with NaN (from lag features)
daily_data = daily_data.dropna()

print(f"Feature engineered dataset shape: {daily_data.shape}")
daily_data.head()

## 7. Prepare Training Data

In [ ]:
# Select features
feature_columns = [
    'DayOfWeek', 'Month', 'Quarter', 'Year', 'DayOfYear', 'WeekOfYear',
    'Quantity', 'Profit',
    'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14', 'Sales_Lag_30',
    'Sales_MA_7', 'Sales_MA_30'
]

X = daily_data[feature_columns]
y = daily_data['Sales']

# Split data (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # Don't shuffle time series data
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 8. Train Models

### Model 1: Linear Regression

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)

# Evaluate
lr_metrics = {
    'train_mae': mean_absolute_error(y_train, lr_train_pred),
    'train_rmse': np.sqrt(mean_squared_error(y_train, lr_train_pred)),
    'train_r2': r2_score(y_train, lr_train_pred),
    'test_mae': mean_absolute_error(y_test, lr_test_pred),
    'test_rmse': np.sqrt(mean_squared_error(y_test, lr_test_pred)),
    'test_r2': r2_score(y_test, lr_test_pred)
}

print("Linear Regression Performance:")
print(f"Train MAE: ${lr_metrics['train_mae']:.2f}")
print(f"Train RMSE: ${lr_metrics['train_rmse']:.2f}")
print(f"Train R²: {lr_metrics['train_r2']:.4f}")
print(f"\nTest MAE: ${lr_metrics['test_mae']:.2f}")
print(f"Test RMSE: ${lr_metrics['test_rmse']:.2f}")
print(f"Test R²: {lr_metrics['test_r2']:.4f}")

### Model 2: Random Forest

In [ ]:
# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predictions
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

# Evaluate
rf_metrics = {
    'train_mae': mean_absolute_error(y_train, rf_train_pred),
    'train_rmse': np.sqrt(mean_squared_error(y_train, rf_train_pred)),
    'train_r2': r2_score(y_train, rf_train_pred),
    'test_mae': mean_absolute_error(y_test, rf_test_pred),
    'test_rmse': np.sqrt(mean_squared_error(y_test, rf_test_pred)),
    'test_r2': r2_score(y_test, rf_test_pred)
}

print("Random Forest Performance:")
print(f"Train MAE: ${rf_metrics['train_mae']:.2f}")
print(f"Train RMSE: ${rf_metrics['train_rmse']:.2f}")
print(f"Train R²: {rf_metrics['train_r2']:.4f}")
print(f"\nTest MAE: ${rf_metrics['test_mae']:.2f}")
print(f"Test RMSE: ${rf_metrics['test_rmse']:.2f}")
print(f"Test R²: {rf_metrics['test_r2']:.4f}")

## 9. Model Comparison

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Test MAE': [lr_metrics['test_mae'], rf_metrics['test_mae']],
    'Test RMSE': [lr_metrics['test_rmse'], rf_metrics['test_rmse']],
    'Test R²': [lr_metrics['test_r2'], rf_metrics['test_r2']]
})

print("\nModel Comparison:")
print(comparison)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Test MAE', 'Test RMSE', 'Test R²']
for idx, metric in enumerate(metrics):
    axes[idx].bar(comparison['Model'], comparison[metric], color=['#667eea', '#764ba2'])
    axes[idx].set_title(metric, fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Value')
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 10. Visualize Predictions

In [ ]:
# Plot actual vs predicted
test_dates = daily_data.iloc[len(X_train):]['Order Date'].values

plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test.values, label='Actual Sales', linewidth=2, color='black')
plt.plot(test_dates, lr_test_pred, label='Linear Regression', linewidth=2, alpha=0.7, color='#667eea')
plt.plot(test_dates, rf_test_pred, label='Random Forest', linewidth=2, alpha=0.7, color='#764ba2')
plt.title('Sales Forecast: Actual vs Predicted', fontsize=16, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Feature Importance (Random Forest)

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='#667eea')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)', fontsize=16, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance.head())

## 12. Save Models

In [ ]:
# Save models
joblib.dump(lr_model, 'linear_regression_model.pkl')
joblib.dump(rf_model, 'random_forest_model.pkl')

# Save feature columns
joblib.dump(feature_columns, 'feature_columns.pkl')

# Save metrics
metrics_summary = {
    'linear_regression': lr_metrics,
    'random_forest': rf_metrics
}
joblib.dump(metrics_summary, 'model_metrics.pkl')

print("✅ Models saved successfully!")
print("Files created:")
print("- linear_regression_model.pkl")
print("- random_forest_model.pkl")
print("- feature_columns.pkl")
print("- model_metrics.pkl")

## 13. Business Insights & Recommendations

In [ ]:
print("="*60)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("="*60)

print("\n1. MODEL PERFORMANCE:")
print(f"   - Random Forest achieved {rf_metrics['test_r2']*100:.1f}% accuracy")
print(f"   - Average prediction error: ${rf_metrics['test_mae']:.2f}")

print("\n2. KEY FINDINGS:")
print(f"   - Most important factor: {feature_importance.iloc[0]['Feature']}")
print(f"   - Average daily sales: ${daily_data['Sales'].mean():.2f}")
print(f"   - Peak sales month: {daily_data.groupby('Month')['Sales'].sum().idxmax()}")

print("\n3. BUSINESS RECOMMENDATIONS:")
print("   ✓ Use forecasts for inventory planning")
print("   ✓ Prepare for seasonal demand variations")
print("   ✓ Monitor prediction errors for continuous improvement")
print("   ✓ Update models monthly with new data")

print("\n4. NEXT STEPS:")
print("   → Deploy models to production (ForecastIQ web app)")
print("   → Set up automated retraining pipeline")
print("   → Create alerts for unusual predictions")
print("   → Build dashboards for stakeholders")

print("\n" + "="*60)

## 🎉 Training Complete!

Download the `.pkl` files and upload them to your ForecastIQ backend for deployment.